In [9]:
from math import exp
import random

def sigmoid(x):
    return 1.0/(1.0+exp(-x))

class Node:
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.upstream = []
        self.output = 0.0
        self.delta = 0.0
    def set_output(self, output):
        self.output = output
    def append_downstream_connection(self, conn):
        self.downstream.append(conn)
    def append_upstream_connection(self, conn):
        self.upstream.append(conn)
    def calc_output(self):
        weighted_sum=sum(
            conn.upstream_node.output * conn.weight
            for conn in self.upstream
        )
        self.output = sigmoid(weighted_sum)
    def calc_output_layer_delta(self, label):
        self.delta = (
            self.output
            *(1-self.output)
            *(label-self.output)
        )
    
    def calc_hidden_layer_delta(self):
        downstream_delta = sum(
            conn.downstream_node.delta * conn.weight
            for conn in self.downstream
        )

        self.delta = (
            self.output
            * (1 - self.output)
            * downstream_delta
        )
class ConstNode:
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.output = 1.0
        self.delta = 0.0
    def append_downstream_connection(self, conn):
        self.downstream.append(conn)

    def calc_hidden_layer_delta(self):
        downstream_delta = sum(
            conn.downstream_node.delta * conn.weight
            for conn in self.downstream
        )

        self.delta = (
            self.output
            * (1 - self.output)
            * downstream_delta
        )

class Connection:
    def __init__(self, upstream_node, downstream_node):
        self.upstream_node = upstream_node
        self.downstream_node = downstream_node

        self.weight = random.uniform(-0.1, 0.1)
        self.gradient = 0.0

    def calc_gradient(self):
        self.gradient = (
            self.downstream_node.delta
            * self.upstream_node.output
        )

    def get_gradient(self):
        return self.gradient

    def update_weight(self, rate):
        self.calc_gradient()
        self.weight += rate * self.gradient

    def __str__(self):
        return (
            f"({self.upstream_node.layer_index}-"
            f"{self.upstream_node.node_index}) -> "
            f"({self.downstream_node.layer_index}-"
            f"{self.downstream_node.node_index}) = "
            f"{self.weight:.6f}"
        )

class Layer:
    def __init__(self, layer_index, node_count):
        self.layer_index = layer_index
        self.nodes = []

        for i in range(node_count):
            self.nodes.append(Node(layer_index, i))

        self.nodes.append(
            ConstNode(layer_index, node_count)
        )

    def set_output(self, data):
        for i in range(len(data)):
            self.nodes[i].set_output(data[i])

    def calc_output(self):
        for node in self.nodes[:-1]:
            node.calc_output()

    def dump(self):
        for node in self.nodes:
            print(
                f"{node.layer_index}-{node.node_index}: "
                f"output={node.output}, delta={node.delta}"
            )

class Connections:
    def __init__(self):
        self.connections = []

    def add_connection(self, connection):
        self.connections.append(connection)

    def dump(self):
        for conn in self.connections:
            print(conn)


class Network:
    def __init__(self, layers):
        self.connections = Connections()
        self.layers = []

        layer_count = len(layers)

        for i in range(layer_count):
            self.layers.append(Layer(i, layers[i]))

        for layer in range(layer_count - 1):
            for upstream_node in self.layers[layer].nodes:
                for downstream_node in self.layers[layer + 1].nodes[:-1]:
                    conn = Connection(
                        upstream_node,
                        downstream_node
                    )

                    self.connections.add_connection(conn)

                    upstream_node.append_downstream_connection(conn)
                    downstream_node.append_upstream_connection(conn)

    def predict(self, sample):
        self.layers[0].set_output(sample)

        for i in range(1, len(self.layers)):
            self.layers[i].calc_output()

        return [
            node.output
            for node in self.layers[-1].nodes[:-1]
        ]
    
    def calc_delta(self, label):
        output_nodes = self.layers[-1].nodes

        for i in range(len(label)):
            output_nodes[i].calc_output_layer_delta(label[i])

        for layer in self.layers[-2::-1]:
            for node in layer.nodes:
                node.calc_hidden_layer_delta()

    def update_weight(self, rate):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.update_weight(rate)

    def train_one_sample(self, label, sample, rate):
        self.predict(sample)
        self.calc_delta(label)
        self.update_weight(rate)

    def train(self, labels, data_set, rate, iteration):
        for i in range(iteration):
            for d in range(len(data_set)):
                self.train_one_sample(
                    labels[d],
                    data_set[d],
                    rate
                )
    
    def calc_gradient(self):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.calc_gradient()

    def get_gradient(self, label, sample):
        self.predict(sample)
        self.calc_delta(label)
        self.calc_gradient()

def network_error(output, label):
    return 0.5 * sum(
        (y - t) ** 2
        for y, t in zip(output, label)
    )

def gradient_check(network, sample_feature, sample_label):
    network.get_gradient(sample_label, sample_feature)

    epsilon = 0.0001
    max_diff = 0.0

    for conn in network.connections.connections:
        actual_gradient = conn.get_gradient()

        original_weight = conn.weight

        conn.weight = original_weight + epsilon

        error1 = network_error(
            network.predict(sample_feature),
            sample_label
        )

        conn.weight = original_weight - epsilon

        error2 = network_error(
            network.predict(sample_feature),
            sample_label
        )

        conn.weight = original_weight

        expected_gradient = (
            (error2 - error1) / (2 * epsilon)
        )

        diff = abs(expected_gradient - actual_gradient)
        max_diff = max(max_diff, diff)

        print(
            f"数值梯度: {expected_gradient:.8f}, "
            f"反向传播: {actual_gradient:.8f}, "
            f"差值: {diff:.10f}"
        )

    print("最大差值:", max_diff)
    return max_diff

if __name__ == "__main__":
    network = Network([2, 3, 1])

    for conn in network.connections.connections:
        conn.weight = 0.1

    sample = [0.6, 0.2]
    label = [1.0]

    max_diff = gradient_check(
        network,
        sample,
        label
    )

    assert max_diff < 1e-6

    print("梯度检查通过！")



             

数值梯度: 0.00158857, 反向传播: 0.00158857, 差值: 0.0000000000
数值梯度: 0.00158857, 反向传播: 0.00158857, 差值: 0.0000000000
数值梯度: 0.00158857, 反向传播: 0.00158857, 差值: 0.0000000000
数值梯度: 0.00052952, 反向传播: 0.00052952, 差值: 0.0000000000
数值梯度: 0.00052952, 反向传播: 0.00052952, 差值: 0.0000000000
数值梯度: 0.00052952, 反向传播: 0.00052952, 差值: 0.0000000000
数值梯度: 0.00264761, 反向传播: 0.00264761, 差值: 0.0000000000
数值梯度: 0.00264761, 反向传播: 0.00264761, 差值: 0.0000000000
数值梯度: 0.00264761, 反向传播: 0.00264761, 差值: 0.0000000000
数值梯度: 0.05817381, 反向传播: 0.05817381, 差值: 0.0000000000
数值梯度: 0.05817381, 反向传播: 0.05817381, 差值: 0.0000000000
数值梯度: 0.05817381, 反向传播: 0.05817381, 差值: 0.0000000000
数值梯度: 0.10676466, 反向传播: 0.10676466, 差值: 0.0000000000
最大差值: 4.4769118967558086e-11
梯度检查通过！


In [1]:
from math import exp
import random


def sigmoid(x):
    return 1.0 / (1.0 + exp(-x))


class Node:
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.upstream = []
        self.output = 0.0
        self.delta = 0.0

    def set_output(self, output):
        self.output = output

    def append_downstream_connection(self, conn):
        self.downstream.append(conn)

    def append_upstream_connection(self, conn):
        self.upstream.append(conn)

    def calc_output(self):
        weighted_sum = sum(
            conn.upstream_node.output * conn.weight
            for conn in self.upstream
        )
        self.output = sigmoid(weighted_sum)

    def calc_output_layer_delta(self, label):
        self.delta = (
            self.output
            * (1 - self.output)
            * (label - self.output)
        )

    def calc_hidden_layer_delta(self):
        downstream_delta = sum(
            conn.downstream_node.delta * conn.weight
            for conn in self.downstream
        )
        self.delta = (
            self.output
            * (1 - self.output)
            * downstream_delta
        )


class ConstNode:
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.output = 1.0
        self.delta = 0.0

    def append_downstream_connection(self, conn):
        self.downstream.append(conn)

    def calc_hidden_layer_delta(self):
        downstream_delta = sum(
            conn.downstream_node.delta * conn.weight
            for conn in self.downstream
        )
        self.delta = (
            self.output
            * (1 - self.output)
            * downstream_delta
        )


class Connection:
    def __init__(self, upstream_node, downstream_node):
        self.upstream_node = upstream_node
        self.downstream_node = downstream_node
        self.weight = random.uniform(-0.1, 0.1)
        self.gradient = 0.0

    def calc_gradient(self):
        self.gradient = (
            self.downstream_node.delta
            * self.upstream_node.output
        )

    def get_gradient(self):
        return self.gradient

    def update_weight(self, rate):
        self.calc_gradient()
        self.weight += rate * self.gradient

    def __str__(self):
        return (
            f"({self.upstream_node.layer_index}-"
            f"{self.upstream_node.node_index}) -> "
            f"({self.downstream_node.layer_index}-"
            f"{self.downstream_node.node_index}) = "
            f"{self.weight:.6f}"
        )


class Layer:
    def __init__(self, layer_index, node_count):
        self.layer_index = layer_index
        self.nodes = []

        for i in range(node_count):
            self.nodes.append(Node(layer_index, i))

        self.nodes.append(ConstNode(layer_index, node_count))

    def set_output(self, data):
        for i in range(len(data)):
            self.nodes[i].set_output(data[i])

    def calc_output(self):
        for node in self.nodes[:-1]:
            node.calc_output()

    def dump(self):
        for node in self.nodes:
            print(
                f"{node.layer_index}-{node.node_index}: "
                f"output={node.output}, delta={node.delta}"
            )


class Connections:
    def __init__(self):
        self.connections = []

    def add_connection(self, connection):
        self.connections.append(connection)

    def dump(self):
        for conn in self.connections:
            print(conn)


class Network:
    def __init__(self, layers):
        self.connections = Connections()
        self.layers = []

        layer_count = len(layers)

        for i in range(layer_count):
            self.layers.append(Layer(i, layers[i]))

        for layer in range(layer_count - 1):
            for upstream_node in self.layers[layer].nodes:
                for downstream_node in self.layers[layer + 1].nodes[:-1]:
                    conn = Connection(upstream_node, downstream_node)
                    self.connections.add_connection(conn)
                    upstream_node.append_downstream_connection(conn)
                    downstream_node.append_upstream_connection(conn)

    def predict(self, sample):
        self.layers[0].set_output(sample)

        for i in range(1, len(self.layers)):
            self.layers[i].calc_output()

        return [node.output for node in self.layers[-1].nodes[:-1]]

    def calc_delta(self, label):
        output_nodes = self.layers[-1].nodes

        for i in range(len(label)):
            output_nodes[i].calc_output_layer_delta(label[i])

        for layer in self.layers[-2::-1]:
            for node in layer.nodes:
                node.calc_hidden_layer_delta()

    def update_weight(self, rate):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.update_weight(rate)

    def train_one_sample(self, label, sample, rate):
        self.predict(sample)
        self.calc_delta(label)
        self.update_weight(rate)

    def train(self, labels, data_set, rate, iteration):
        for i in range(iteration):
            for d in range(len(data_set)):
                self.train_one_sample(labels[d], data_set[d], rate)

    # 保留前面梯度检查用到的两个方法
    def calc_gradient(self):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.calc_gradient()

    def get_gradient(self, label, sample):
        self.predict(sample)
        self.calc_delta(label)
        self.calc_gradient()


class Loader:
    def __init__(self, path, count):
        self.path = path
        self.count = count

    def get_file_content(self):
        with open(self.path, "rb") as f:
            content = f.read()
        return content

    def to_int(self, byte):
        return byte


class ImageLoader(Loader):
    def get_picture(self, content, index):
        start = index * 28 * 28 + 16
        picture = []

        for i in range(28):
            picture.append([])
            for j in range(28):
                picture[i].append(
                    self.to_int(content[start + i * 28 + j])
                )

        return picture

    def get_one_sample(self, picture):
        sample = []
        for i in range(28):
            for j in range(28):
                sample.append(picture[i][j] / 255.0)
        return sample

    def load(self):
        content = self.get_file_content()
        data_set = []

        for index in range(self.count):
            data_set.append(
                self.get_one_sample(
                    self.get_picture(content, index)
                )
            )

        return data_set


class LabelLoader(Loader):
    def load(self):
        content = self.get_file_content()
        labels = []

        for index in range(self.count):
            labels.append(self.norm(content[index + 8]))

        return labels

    def norm(self, label):
        label_vec = []
        label_value = self.to_int(label)

        for i in range(10):
            if i == label_value:
                label_vec.append(0.9)
            else:
                label_vec.append(0.1)

        return label_vec


def get_training_data_set(count=60000):
    image_loader = ImageLoader(
        "data/train-images-idx3-ubyte",
        count
    )
    label_loader = LabelLoader(
        "data/train-labels-idx1-ubyte",
        count
    )
    return image_loader.load(), label_loader.load()


def get_test_data_set(count=10000):
    image_loader = ImageLoader(
        "data/t10k-images-idx3-ubyte",
        count
    )
    label_loader = LabelLoader(
        "data/t10k-labels-idx1-ubyte",
        count
    )
    return image_loader.load(), label_loader.load()


def get_result(vec):
    max_value_index = 0
    max_value = 0

    for i in range(len(vec)):
        if vec[i] > max_value:
            max_value = vec[i]
            max_value_index = i

    return max_value_index


def evaluate(network, test_data_set, test_labels):
    error = 0
    total = len(test_data_set)

    for i in range(total):
        label = get_result(test_labels[i])
        predict = get_result(network.predict(test_data_set[i]))
        if label != predict:
            error += 1

    return float(error) / float(total)


def train_and_evaluate():
    train_data_set, train_labels = get_training_data_set(200)
    test_data_set, test_labels = get_test_data_set(50)

    network = Network([784, 20, 10])

    for epoch in range(1, 4):
        network.train(
            train_labels,
            train_data_set,
            0.3,
            1
        )

        error_ratio = evaluate(
            network,
            test_data_set,
            test_labels
        )

        print(
            f"第 {epoch} 轮训练完成，"
            f"错误率：{error_ratio:.2%}，"
            f"准确率：{1 - error_ratio:.2%}"
        )

    real_number = get_result(test_labels[0])
    predicted_number = get_result(
        network.predict(test_data_set[0])
    )

    print("第一张测试图片的真实数字:", real_number)
    print("神经网络预测的数字:", predicted_number)

    return network


if __name__ == "__main__":
    network = train_and_evaluate()


第 1 轮训练完成，错误率：90.00%，准确率：10.00%
第 2 轮训练完成，错误率：66.00%，准确率：34.00%
第 3 轮训练完成，错误率：40.00%，准确率：60.00%
第一张测试图片的真实数字: 7
神经网络预测的数字: 7
